# MODUL 1
## Dasar-Dasar Logika Fuzzy dan Fuzzifikasi

**Mata kuliah:** Logika Fuzzy  
**Bentuk:** Praktikum berbasis Python/Jupyter Notebook  
**Alokasi:** 5 pertemuan

## Tujuan Modul
Mahasiswa mampu memahami konsep dasar logika fuzzy dan mengubah permasalahan nyata menjadi representasi fuzzy menggunakan variabel linguistik dan fungsi keanggotaan.

## Prasyarat dan aturan kerja
- Memahami operasi dasar Python, fungsi, array NumPy, dan visualisasi Matplotlib.
- Jalankan sel secara berurutan. Amati grafik dan tuliskan interpretasi pada sel jawaban.
- Instalasi `scikit-fuzzy` hanya diperlukan untuk bagian integrasi pada Pertemuan 3.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=3, suppress=True)
print('Lingkungan praktikum siap.')

# Pertemuan 1 — Pengantar Logika Fuzzy

## 1.1 Mengambil keputusan dalam ketidakpastian
Banyak keputusan tidak memiliki batas yang tegas. Waktu respons 3,9 detik mungkin masih dianggap cepat oleh satu pengguna, tetapi lambat oleh pengguna lain. Logika crisp memaksa nilai masuk ke kategori benar/salah, sedangkan logika fuzzy menyatakan **seberapa kuat** suatu nilai termasuk ke dalam kategori.

Pada logika Boolean/crisp, sebuah objek hanya memiliki keanggotaan 0 atau 1. Pada logika fuzzy, derajat keanggotaan bernilai kontinu pada interval $[0, 1]$. Nilai 0,8 berarti objek cukup kuat memenuhi konsep tersebut, bukan probabilitas bahwa objek termasuk.

### Sejarah singkat
Konsep fuzzy diperkenalkan Lotfi A. Zadeh pada 1965 melalui teori himpunan fuzzy. Perkembangannya meliputi pengendali fuzzy, sistem pakar, pengambilan keputusan multikriteria, klasifikasi, hingga pembelajaran mesin.

### Penerapan pada Teknologi Informasi
Fuzzy digunakan pada sistem rekomendasi, penilaian kualitas layanan, diagnosis, klasifikasi citra, pengaturan jaringan, prioritas tiket helpdesk, dan sistem pendukung keputusan. Contoh sistem rekomendasi dapat menggabungkan tingkat kepuasan dan waktu respons untuk menghasilkan rekomendasi layanan.

In [ ]:
# Representasi crisp dan fuzzy untuk kualitas layanan
ratings = np.array([2, 3, 4, 5])
crisp_good = (ratings >= 4).astype(int)
fuzzy_good = np.array([0.0, 0.25, 0.75, 1.0])

print('Rating       :', ratings)
print('Crisp baik   :', crisp_good)
print('Fuzzy baik   :', fuzzy_good)

plt.figure(figsize=(7, 3.5))
plt.step(ratings, crisp_good, where='mid', label='Crisp: baik/tidak baik')
plt.plot(ratings, fuzzy_good, 'o-', label='Fuzzy: derajat baik')
plt.xlabel('Rating layanan')
plt.ylabel('Derajat keanggotaan')
plt.ylim(-0.05, 1.05)
plt.legend()
plt.show()

**Latihan 1.** Jelaskan mengapa batas crisp `rating >= 4` dapat menghasilkan keputusan yang kurang halus. Berikan satu contoh kondisi nyata yang cocok dimodelkan dengan fuzzy.

# Pertemuan 2 — Himpunan Fuzzy dan Variabel Linguistik

## 2.1 Konsep inti
- **Himpunan crisp:** setiap elemen memiliki keanggotaan 0 atau 1.
- **Himpunan fuzzy:** setiap elemen $x$ memiliki derajat keanggotaan $\mu_A(x) \in [0,1]$.
- **Semesta pembicaraan:** seluruh nilai yang mungkin, misalnya waktu respons 0–10 detik.
- **Domain:** rentang nilai yang digunakan oleh satu label fuzzy.
- **Variabel linguistik:** variabel yang nilainya berupa label bahasa, misalnya `Waktu Respons = cepat`, `sedang`, atau `lambat`.

Label linguistik tidak harus saling eksklusif. Satu nilai dapat memiliki keanggotaan pada beberapa label sekaligus. Inilah yang memungkinkan transisi bertahap.

In [ ]:
def triangular(x, left, center, right):
    rising = (x - left) / (center - left) if center != left else 1.0
    falling = (right - x) / (right - center) if right != center else 1.0
    return np.maximum(0, np.minimum(rising, falling))

response = np.linspace(0, 10, 501)
fast = np.clip((5 - response) / 5, 0, 1)
moderate = triangular(response, 2, 5, 8)
slow = np.clip((response - 5) / 5, 0, 1)

linguistic_variable = {
    'nama': 'Waktu Respons',
    'semesta_pembicaraan': (0, 10),
    'unit': 'detik',
    'label': {'cepat': fast, 'sedang': moderate, 'lambat': slow}
}
print('Variabel:', linguistic_variable['nama'])
print('Label:', list(linguistic_variable['label']))

plt.figure(figsize=(8, 4))
for label, membership in linguistic_variable['label'].items():
    plt.plot(response, membership, label=label.title())
plt.xlabel('Waktu respons (detik)')
plt.ylabel('Derajat keanggotaan')
plt.ylim(0, 1.05)
plt.title('Variabel linguistik: Waktu Respons')
plt.legend()
plt.show()

In [ ]:
# Fuzzifikasi sederhana: mencari derajat tiap label pada satu nilai input
input_response = 6.0
degrees = {label: float(np.interp(input_response, response, membership))
           for label, membership in linguistic_variable['label'].items()}
print(f'Waktu respons = {input_response:.1f} detik')
for label, degree in degrees.items():
    print(f'  {label:>6}: {degree:.2f}')

**Latihan 2.** Ubah semesta waktu respons menjadi 0–20 detik dan tentukan kembali domain label `cepat`, `sedang`, dan `lambat`. Fuzzifikasikan input 7 detik serta jelaskan label yang paling dominan.

# Pertemuan 3 — Fungsi Keanggotaan

Fungsi keanggotaan memetakan nilai input ke derajat keanggotaan pada $[0,1]$. Pemilihannya mengikuti bentuk data dan makna domain, bukan sekadar memilih grafik yang terlihat menarik.

- **Segitiga:** sederhana dan cocok untuk konsep yang memiliki satu titik puncak.
- **Trapesium:** cocok ketika ada rentang nilai yang sepenuhnya memenuhi label.
- **Gaussian:** transisi halus di sekitar pusat.
- **Sigmoid:** cocok untuk konsep yang meningkat atau menurun secara bertahap.

In [ ]:
def trapezoidal(x, a, b, c, d):
    return np.maximum(0, np.minimum(np.minimum((x-a)/(b-a), 1), (d-x)/(d-c)))

def gaussian(x, center, sigma):
    return np.exp(-0.5 * ((x-center)/sigma) ** 2)

def sigmoid(x, center, slope):
    return 1 / (1 + np.exp(-slope * (x-center)))

x = np.linspace(0, 10, 501)
membership_functions = {
    'Segitiga': triangular(x, 2, 5, 8),
    'Trapesium': trapezoidal(x, 2, 4, 6, 8),
    'Gaussian': gaussian(x, 5, 1.4),
    'Sigmoid': sigmoid(x, 5, 1.2)
}

fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True, sharey=True)
for axis, (name, membership) in zip(axes.ravel(), membership_functions.items()):
    axis.plot(x, membership, linewidth=2)
    axis.set_title(name)
    axis.set_xlabel('Nilai input')
    axis.set_ylabel('Derajat keanggotaan')
    axis.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Integrasi scikit-fuzzy (opsional)
try:
    import skfuzzy as fuzz
    skfuzzy_triangle = fuzz.trimf(x, [2, 5, 8])
    skfuzzy_trapezoid = fuzz.trapmf(x, [2, 4, 6, 8])
    print('scikit-fuzzy tersedia; contoh trimf dan trapmf berhasil dibuat.')
except ImportError:
    print('scikit-fuzzy belum terpasang. Jalankan: %pip install scikit-fuzzy')

**Latihan 3.** Bandingkan segitiga dan Gaussian untuk label `kepuasan sedang`. Fungsi mana yang menghasilkan transisi paling halus? Jelaskan dampaknya terhadap proses fuzzifikasi.

# Pertemuan 4 — Operasi Himpunan Fuzzy

Operasi dasar menggunakan derajat keanggotaan, bukan hanya keanggotaan biner. Operasi standar yang digunakan adalah:
- **Union / OR:** $\mu_{A \cup B}(x)=\max(\mu_A(x),\mu_B(x))$.
- **Intersection / AND:** $\mu_{A \cap B}(x)=\min(\mu_A(x),\mu_B(x))$.
- **Complement:** $\mu_{\neg A}(x)=1-\mu_A(x)$.

Minimum merupakan contoh **T-norm**, sedangkan maksimum merupakan contoh **T-conorm**. Keduanya dapat diganti dengan operator lain sesuai kebutuhan sistem.

In [ ]:
quality = np.array([0.2, 0.7, 0.9, 0.4])
speed = np.array([0.8, 0.5, 0.6, 0.3])
operations = {
    'Union (OR)': np.maximum(quality, speed),
    'Intersection (AND)': np.minimum(quality, speed),
    'Complement kualitas': 1 - quality,
    'Product T-norm': quality * speed,
    'Probabilistic sum T-conorm': quality + speed - quality * speed
}

print(' A       =', quality)
print(' B       =', speed)
for name, result in operations.items():
    print(f'{name:28} = {result}')

indices = np.arange(len(quality))
plt.figure(figsize=(9, 4.5))
plt.plot(indices, quality, 'o-', label='Kualitas')
plt.plot(indices, speed, 'o-', label='Kecepatan')
plt.plot(indices, operations['Intersection (AND)'], 's--', label='AND: min')
plt.plot(indices, operations['Union (OR)'], 'd--', label='OR: max')
plt.xticks(indices, [f'Data {i+1}' for i in indices])
plt.ylim(0, 1.05)
plt.ylabel('Derajat keanggotaan')
plt.title('Perbandingan operasi fuzzy')
plt.legend()
plt.show()

**Latihan 4.** Ubah nilai anggota kedua himpunan, lalu hitung ulang semua operasi. Mengapa hasil AND tidak pernah lebih besar daripada salah satu operand? Kapan operator produk lebih ketat daripada `min`?

# Pertemuan 5 — Asesmen Modul 1
## Final Practical Assignment 1: Fuzzy Modeling

Pilih satu permasalahan sederhana dari bidang TI atau kehidupan sehari-hari, misalnya prioritas tiket helpdesk, kualitas jaringan, kelayakan rekomendasi film, atau tingkat risiko transaksi. Dokumentasikan delapan langkah berikut.

1. Menentukan permasalahan dan alasan fuzzy diperlukan.
2. Menentukan variabel input dan output.
3. Menentukan domain masing-masing variabel.
4. Menentukan himpunan/label linguistik.
5. Menentukan fungsi keanggotaan dan alasan pemilihannya.
6. Mengimplementasikan fungsi keanggotaan menggunakan Python.
7. Membuat visualisasi yang diberi judul, label sumbu, dan legenda.
8. Melakukan fuzzifikasi terhadap minimal lima data input dan menganalisis hasilnya.

### Kriteria penilaian
| Komponen | Bobot |
|---|---:|
| Perumusan masalah dan variabel | 20% |
| Domain, label, dan fungsi keanggotaan | 25% |
| Implementasi Python | 25% |
| Visualisasi | 15% |
| Analisis fuzzifikasi dan kerapian laporan | 15% |

In [ ]:
# Template tugas: lengkapi atau modifikasi sesuai permasalahan pilihan Anda
problem_name = 'Prioritas tiket helpdesk'
input_name = 'Waktu respons (menit)'
input_domain = np.linspace(0, 120, 601)
labels = {
    'cepat': np.clip((60 - input_domain) / 60, 0, 1),
    'sedang': triangular(input_domain, 30, 60, 90),
    'lambat': np.clip((input_domain - 60) / 60, 0, 1)
}

sample_inputs = np.array([10, 35, 55, 75, 110])
fuzzified = {label: np.interp(sample_inputs, input_domain, membership)
             for label, membership in labels.items()}

print('Permasalahan:', problem_name)
print('Variabel input:', input_name)
print('Derajat keanggotaan:')
for label, values in fuzzified.items():
    print(f'  {label:>6}: {values}')

plt.figure(figsize=(9, 4))
for label, membership in labels.items():
    plt.plot(input_domain, membership, label=label.title())
plt.scatter(sample_inputs, np.zeros_like(sample_inputs), color='black', zorder=3, label='Data uji')
plt.xlabel(input_name)
plt.ylabel('Derajat keanggotaan')
plt.ylim(-0.05, 1.05)
plt.title(f'Model fuzzy: {problem_name}')
plt.legend()
plt.show()

## Format pengumpulan
Kumpulkan notebook yang sudah dijalankan beserta interpretasi tertulis. Setiap keputusan pemodelan harus dapat ditelusuri dari karakteristik masalah dan data. Pastikan grafik tidak kosong, semua label terbaca, dan hasil fuzzifikasi dibahas untuk setiap data uji.

## Refleksi akhir
1. Apa perbedaan utama antara nilai fuzzy dan probabilitas?
2. Bagaimana perubahan parameter fungsi keanggotaan mengubah keputusan sistem?
3. Apa risiko memilih domain atau label yang terlalu sempit?

## Instalasi dependensi (jalankan sekali)

Jika pustaka belum tersedia di environment Anda, hapus tanda komentar pada sel berikut dan jalankan sekali. Setelah itu restart kernel bila diminta.

In [ ]:
# Jalankan jika NumPy/Matplotlib belum tersedia.
# %pip install numpy matplotlib scikit-fuzzy
print('Aktifkan perintah %pip di atas bila dependensi belum terpasang.')